In [1]:
include("../RayTracing.jl")
using Colors, FileIO, ImageMagick

In [2]:
struct Atmosphere
    sun_direction::RayTracing.Vec3
    earth_radius::Float64
    atmosphere_radius::Float64
    Hr::Float64
    Hm::Float64
    beta_R::RayTracing.Vec3
    beta_M::RayTracing.Vec3
    function Atmosphere(
        sun_direction=RayTracing.Vec3(0,1,0), 
        earth_radius=6360e3, 
        atmosphere_radius=6420e3, 
        Hr=7994, 
        Hm=1200,
        beta_R=RayTracing.Vec3(3.8e-6, 13.5e-6, 33.1e-6),
        beta_M=RayTracing.Vec3(21e-6, 21e-6, 21e-6)
    )
        return new(sun_direction, earth_radius, atmosphere_radius, Hr, Hm, beta_R, beta_M)
    end
end

In [3]:
function Base.:*(a::RayTracing.Vec3, b::RayTracing.Vec3)
    return RayTracing.Vec3(a.x * b.x, a.y * b.y, a.z * b.z)
end

In [4]:
function render_sky_dome(sun_dir::RayTracing.Vec3)
    atmosphere = Atmosphere(sun_dir)
    width = 512
    height = 512
    image = zeros(RGB, 512, 512)
    for j in 0:(height-1)
        y = 2.0 * (j + 0.5) / (height - 1) - 1.0
        for i in 0:(width-1)
            x = 2.0 * (i + 0.5) / (width - 1) - 1.0 
            z2 = x^2 + y^2
            if z2 <= 1.0 
                phi = atan(y, x) 
                theta = acos(1 - z2) 
                dir = RayTracing.Vec3(sin(theta) * cos(phi), cos(theta), sin(theta) * sin(phi))
                L = compute_incident_light(
                    atmosphere,
                    RayTracing.Vec3(0, atmosphere.earth_radius + 1, 0), 
                    dir, 
                    0.0, 
                    typemax(Float64)
                ) 
                image[j+1,i+1] = RGB(L.x, L.y, L.z)
            end
        end
    end
    return image
end

function solve_quadratic(a::Float64, b::Float64, c::Float64)::Tuple{Bool, Float64, Float64}
    # Find disriminant.
    d = b ^ 2 - 4 * a * c
    if d < 0
        return false, typemax(Float64), typemax(Float64)
    end
    d = d |> sqrt
    # Compute roots.
    q = -0.5 * (b + (b < 0 ? -d : d))
    t0 = q / a
    t1 = c / q
    if t0 > t1
        t0, t1 = t1, t0
    end
    return true, t0, t1
end

function ray_sphere_intersect(
    orig::RayTracing.Vec3,
    dir::RayTracing.Vec3,
    radius::Float64
)::Tuple{Bool, Float64, Float64}

    A = dir.x * dir.x + dir.y * dir.y + dir.z * dir.z
    B = 2 * (dir.x * orig.x + dir.y * orig.y + dir.z * orig.z)
    C = orig.x * orig.x + orig.y * orig.y + orig.z * orig.z - radius * radius

    check, t0, t1 = solve_quadratic(A, B, C)
    if !check 
        return false, typemax(Float64), typemax(Float64)
    end

    if (t0 > t1) 
        t0, t1 = t1, t0
    end

    return true, t0, t1
end

function compute_incident_light(atmosphere::Atmosphere, orig::RayTracing.Vec3, dir::RayTracing.Vec3, tmin::Float64, tmax::Float64)
    check, t0, t1 = ray_sphere_intersect(orig, dir, atmosphere.atmosphere_radius)
    if !check || t1 < 0
        return 0.0
    end
    if t0 > tmin && t0 > 0
        tmin = t0
    end
    if t1 < tmax
        tmax = t1
    end
    num_samples = 16
    num_samples_light = 8
    segment_length = (tmax - tmin) / num_samples
    t_current = tmin
    sum_R = RayTracing.Vec3(0)
    sum_M = RayTracing.Vec3(0)
    optical_depth_R = 0.0
    optical_depth_M = 0.0
    mu = RayTracing.dot(dir, atmosphere.sun_direction)
    phase_R = 3.0 / (16.0 * pi) * (1 + mu^2)
    g = 0.76
    phase_M = 3.0 / (8.0 * pi) * ((1.0 - g^2) * (1.0 + mu^2)) / ((2.0 + g^2) * (1.0 + g^2 - 2.0 * g * mu)^1.5)
    for i in 0:(num_samples-1)
        sample_position = orig + (t_current + segment_length * 0.5) * dir
        height = RayTracing.length_pbrt(sample_position) - atmosphere.earth_radius
        hr = exp(-height / atmosphere.Hr) * segment_length
        hm = exp(-height / atmosphere.Hm) * segment_length 
        optical_depth_R += hr 
        optical_depth_M += hm 
        check, t0light, t1light = ray_sphere_intersect(
            sample_position, 
            atmosphere.sun_direction, 
            atmosphere.atmosphere_radius
        )
        segment_length_light = t1light / num_samples_light
        t_current_light = 0
        optical_depth_light_R = 0
        optical_depth_light_M = 0 
        for j in 0:(num_samples_light-1)
            sample_position_light = sample_position + (t_current_light + segment_length_light * 0.5) * atmosphere.sun_direction
            height_light = RayTracing.length_pbrt(sample_position_light) - atmosphere.earth_radius
            if height_light < 0
                break
            end
            optical_depth_light_R += exp(-height_light / atmosphere.Hr) * segment_length_light 
            optical_depth_light_M += exp(-height_light / atmosphere.Hm) * segment_length_light 
            t_current_light += segment_length_light
        end
        tau = atmosphere.beta_R * (optical_depth_R + optical_depth_light_R) + atmosphere.beta_M * 1.1 * (optical_depth_M + optical_depth_light_M)
        attenuation = RayTracing.Vec3(exp(-tau.x), exp(-tau.y), exp(-tau.z))
        sum_R += attenuation * hr
        sum_M += attenuation * hm 
        t_current += segment_length
    end
    return (sum_R * atmosphere.beta_R * phase_R + sum_M * atmosphere.beta_M * phase_M) * 20 
end

compute_incident_light (generic function with 1 method)

In [5]:
# nangles = 128
# gif = zeros(RGB, 512, 512, nangles)
# for i in 0:(nangles-1)
#     angle = i / (nangles-1) * pi * 0.6
#     v = RayTracing.Vec3(0, cos(angle), -sin(angle))
#     gif[:, :, i+1] = map(RayTracing.clamp01nan, render_sky_dome(v))
#     print("frame $(i+1) of $(nangles) done!\n")
# end

# save("sky1.gif", gif)

In [7]:
n_angles = 80
width = 512
height = 512
aspectRatio = width / height
fov = 65.0 
angle = tan(fov * pi / 180 * 0.5)
numPixelSamples = 4

gif = zeros(RGB, width, height, n_angles)
for a in 0:(n_angles-1)
    image = zeros(RGB, width, height)
    angle = a / (n_angles-1) * pi * 0.6
    v = RayTracing.Vec3(0, cos(angle), -sin(angle))
    atmosphere = Atmosphere(v)
    orig = RayTracing.Vec3(0, atmosphere.earth_radius + 1000, 30000)
    for y in 0:(height-1)
        for x in 0:(width-1)
            accum = RayTracing.Vec3(0)
            for m in 0:(numPixelSamples-1)
                for n in 0:(numPixelSamples-1)
                    rayx = (2 * (x + (m + RayTracing.rand()) / numPixelSamples) / width - 1) * aspectRatio * angle
                    rayy = (1 - (y + (n + RayTracing.rand()) / numPixelSamples) / height * 2) * angle
                    dir = RayTracing.Vec3(rayx, rayy, -1) 
                    dir = RayTracing.normalize(dir) 
                    check, t0, t1 = ray_sphere_intersect(orig, dir, atmosphere.earth_radius)
                    tMax = typemax(Float64)
                    if (check && t1 > 0) 
                        tMax = max(0.0, t0)
                    end
                    accum += compute_incident_light(atmosphere, orig, dir, 0.0, tMax)
                end
            end
            tmp = accum / numPixelSamples^2
            image[y+1,x+1] = RGB(tmp.x, tmp.y, tmp.z)
        end
    end
    print("frame $(a+1) of $(n_angles) done!\n")
    gif[:,:,a+1] = image
end
save("sky2.gif", map(RayTracing.clamp01nan, gif))

frame 1 of 80 done!
frame 2 of 80 done!
frame 3 of 80 done!
frame 4 of 80 done!
frame 5 of 80 done!
frame 6 of 80 done!
frame 7 of 80 done!
frame 8 of 80 done!
frame 9 of 80 done!
frame 10 of 80 done!
frame 11 of 80 done!
frame 12 of 80 done!
frame 13 of 80 done!
frame 14 of 80 done!
frame 15 of 80 done!
frame 16 of 80 done!
frame 17 of 80 done!
frame 18 of 80 done!
frame 19 of 80 done!
frame 20 of 80 done!
frame 21 of 80 done!
frame 22 of 80 done!
frame 23 of 80 done!
frame 24 of 80 done!
frame 25 of 80 done!
frame 26 of 80 done!
frame 27 of 80 done!
frame 28 of 80 done!
frame 29 of 80 done!
frame 30 of 80 done!
frame 31 of 80 done!
frame 32 of 80 done!
frame 33 of 80 done!
frame 34 of 80 done!
frame 35 of 80 done!
frame 36 of 80 done!
frame 37 of 80 done!
frame 38 of 80 done!
frame 39 of 80 done!
frame 40 of 80 done!
frame 41 of 80 done!
frame 42 of 80 done!
frame 43 of 80 done!
frame 44 of 80 done!
frame 45 of 80 done!
frame 46 of 80 done!
frame 47 of 80 done!
frame 48 of 80 done!
f